# Application du solver au dataset

## Breast Cancer

In [ ]:
import numpy as np
from gurobipy import Model, GRB, quicksum

TARGET = "Benign"
X = df_BC.drop(columns=[TARGET])

beta = logreg.coef_[0]
features = list(X.columns)

# Choisir x (préduit 1) et y (préduit 0)
pred = logreg.predict(X)
ix = X.index[pred == 1][0]
iy = X.index[pred == 0][0]
x = X.loc[ix]
y_ = X.loc[iy]

# Vérifier p(x) > p(y) et s(x) > s(y)
px = float(logreg.predict_proba([x])[0, 1])
py = float(logreg.predict_proba([y_])[0, 1])
sx = float(logreg.decision_function([x])[0])
sy = float(logreg.decision_function([y_])[0])

print(f"x index={ix} | p(x)={px:.4f} | s(x)={sx:.4f}")
print(f"y index={iy} | p(y)={py:.4f} | s(y)={sy:.4f}")
assert sx > sy, "On veut comparer un x strictement préféré à y (s(x) > s(y))."

# Contributions Δ_j = β_j (x_j - y_j)
delta = {f: float(beta[i] * (x[f] - y_[f])) for i, f in enumerate(features)}
pros = [f for f in features if delta[f] > 0]
cons = [f for f in features if delta[f] < 0]
print("s(x)-s(y) =", sum(delta.values()))
print("pros:", pros)
print("cons:", cons)

# Exemple : explication (1-1) par MILP
T = [(p, c) for p in pros for c in cons if delta[p] + delta[c] > 0]

m = Model("explain_1-1_realdata"); m.params.OutputFlag = 0
z = m.addVars(T, vtype=GRB.BINARY)

for c in cons:
    m.addConstr(quicksum(z[p, c] for p in pros if (p, c) in z) == 1)

m.setObjective(quicksum(z[p, c] for (p, c) in T), GRB.MINIMIZE)
m.optimize()

if m.Status == GRB.OPTIMAL:
    E = [(p, c) for (p, c) in T if z[p, c].X > 0.5]
    print("Explication (1-1):", E)
else:
    print("Pas d'explication (1-1) (infeasible).")
